In [ ]:
import pandas as pd
import numpy as np
import re
from tqdm import tqdm
import os
import importlib
import time
import itertools
import datetime
# import IPython
# import sys
import ipynbname

from kfirlib.comparisons import *
from kfirlib.util import *
from kfirlib import pubmed, gpt

In [ ]:
importlib.reload(pubmed)

In [ ]:
DIR = r'C:\Users\israe\OneDrive\Documents\post doc\kfir\covid\tables'
IN_PATH = os.path.join(DIR, '549_results_from_PUBMED.xlsx')
EXTRA_IN_PATH = os.path.join(DIR, 'deduplicate_with_eligibility_2 - imp.xlsx')
OUT_PATH = os.path.join(DIR, '549_results_from_PUBMED_with_eligibility_all.xlsx')
LOG_PATH = os.path.join(DIR, 'log.txt')

In [ ]:
orig_df = pd.read_excel(IN_PATH)
orig_df.columns

In [ ]:
orig_df.rename(columns={'Publication Year': 'Year'}, inplace=True)

In [ ]:
filled_in = pubmed.add_lines_by_pubmed_id(orig_df, 'PMID')
filled_in.columns

In [ ]:
eligibility_abstract = gpt.check_eligibility(filled_in, criteria=gpt.PAUL_CRITERIA)
eligibility_abstract.columns

In [ ]:
eligibility_abstract.rename(columns={'reasoning': 'reasoning_title_abstract', 'decision': 'decision_title_abstract'}, inplace=True)
eligibility_abstract.columns

In [ ]:
no_abstract = eligibility_abstract.copy()
no_abstract.loc['abstract', :] = np.nan
no_abstract_res = gpt.check_eligibility(no_abstract, criteria=gpt.PAUL_CRITERIA, batch_size=100)

In [ ]:
part = no_abstract_res[['reasoning', 'decision']]
part = part.rename(columns={'reasoning': 'reasoning_title', 'decision': 'decision_title'})
eligibility_all = pd.concat([eligibility_abstract, part], axis=1)
eligibility_all.head()

In [ ]:
eligibility_all.decision_title.isna().sum(), eligibility_all.decision_title_abstract.isna().sum()

In [ ]:
eligibility_all.decision_title.mean(), eligibility_all.decision_title_abstract.mean(), (eligibility_all.decision_title != eligibility_all.decision_title_abstract).mean()

In [ ]:
eligibility_all.to_excel(OUT_PATH, index=False)

## Confusion matrix

In [ ]:
eligibility_all = pd.read_excel(OUT_PATH)
eligibility_all.columns

In [ ]:
big_df = pd.read_excel(EXTRA_IN_PATH)
big_df.columns

In [ ]:
eligibility_all[['Title', 'title']].isna().sum()

In [ ]:
eligibility_all = eligibility_all[~eligibility_all.Title.isna()]

In [ ]:
eligibility_all['title_extract'] = eligibility_all.Title.apply(extract_letters)

In [ ]:
eligibility_all.title_extract.isin(big_df.title_extract).mean()

In [ ]:
eligibility_all.title_extract.dtype

In [ ]:
eligibility_all = pd.merge(eligibility_all, big_df[['title_extract', 'is_in_paul']], on='title_extract')

In [ ]:
def confusion_matrix(decision, gt):
    df = pd.DataFrame({'decision': decision, 'truth': gt})
    df['val'] = 1
    res = df.groupby(['decision', 'truth']).sum().reset_index(drop=False)
    return res.pivot(index='decision', columns='truth', values='val').fillna(0)

In [ ]:
log_file = open(LOG_PATH, 'at')

In [ ]:
log_file.write(f'\n==================== {datetime.datetime.now().strftime("%Y-%m-%d %H:%M")} ====================\n\n')
log_file.write(f'{ipynbname.path()}\n')

In [ ]:
log_file.write(f'Input: {IN_PATH}\n')
log_file.write(f'Extra Input: {EXTRA_IN_PATH}\n')
log_file.write(f'Output: {OUT_PATH}\n')

In [ ]:
log_file.write(f'\nSystem prompt:\n\n{gpt.SYSTEM_PROMPT}\n\n')
log_file.write(f'\nCriteria:\n\n{gpt.PAUL_CRITERIA}\n\n')

Title

In [ ]:
confusion = confusion_matrix(decision=eligibility_all.decision_title, gt=eligibility_all.is_in_paul)
confusion

In [ ]:
log_file.write(f'\nConfusion for titles:\n{confusion}\n\n')

Title and abstract

In [ ]:
confusion = confusion_matrix(decision=eligibility_all.decision_title_abstract == 1, gt=eligibility_all.is_in_paul)
confusion

In [ ]:
log_file.write(f'\nConfusion for titles and abstracts:\n{confusion}\n\n')

AND

In [ ]:
confusion = confusion_matrix(decision=(eligibility_all.decision_title_abstract == 1) & eligibility_all.decision_title, gt=eligibility_all.is_in_paul)

In [ ]:
log_file.write(f'\nConfusion for AND:\n{confusion}\n\n')

In [ ]:
log_file.close()